In [0]:
%pip install duckdb==1.5.3

In [0]:
%restart_python

In [ ]:
import datetime
    
CATALOG  = "tabular"
SCHEMA   = "dataexpert"
VOLUME   = "/Volumes/tabular/dataexpert/josh_wafflehouz/transit_performance/md_export"
MD_TOKEN = dbutils.secrets.get(scope="transit-api", key="MOTHERDUCK_TOKEN")

yesterday  = datetime.date.today() - datetime.timedelta(days=1)
cutoff_90d = str(yesterday - datetime.timedelta(days=89))
cutoff_28d = str(yesterday - datetime.timedelta(days=27))
yesterday  = str(yesterday)

print("90d window:", cutoff_90d, "->", yesterday)
print("28d window:", cutoff_28d, "->", yesterday)

TABLES = [
    ("gold_stop_dwell_fact",          f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_stop_dwell_fact WHERE service_date >= '{cutoff_90d}'"),
    ("gold_stop_dwell_inferred",      f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_stop_dwell_inferred WHERE service_date >= '{cutoff_90d}'"),
    ("gold_rail_stop_actuals",        f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_rail_stop_actuals WHERE service_date >= '{cutoff_90d}'"),
    ("gold_route_metrics_15min",      f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_route_metrics_15min WHERE service_date >= '{cutoff_90d}'"),
    ("gold_missed_trips",             f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_missed_trips WHERE service_date >= '{cutoff_90d}'"),
    ("gold_anomaly_events",           f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_anomaly_events WHERE service_date >= '{cutoff_90d}'"),
    ("gold_trip_timeline_fact",       f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_trip_timeline_fact WHERE service_date >= '{cutoff_90d}'"),
    ("gold_trip_path_fact",           f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_trip_path_fact WHERE service_date >= '{cutoff_28d}'"),
    ("gold_route_segment_congestion", f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_route_segment_congestion WHERE service_date >= '{cutoff_28d}'"),
    ("silver_fact_vehicle_positions",  f"SELECT * FROM {CATALOG}.{SCHEMA}.silver_fact_vehicle_positions WHERE service_date >= '{cutoff_28d}'"),
    ("silver_dim_route",              f"SELECT * FROM {CATALOG}.{SCHEMA}.silver_dim_route"),
    ("silver_dim_stop",               f"SELECT * FROM {CATALOG}.{SCHEMA}.silver_dim_stop"),
    ("silver_dim_trip",               f"SELECT * FROM {CATALOG}.{SCHEMA}.silver_dim_trip"),
    ("silver_fact_stop_schedule",     f"SELECT * FROM {CATALOG}.{SCHEMA}.silver_fact_stop_schedule"),
    ("silver_fact_shape_points",      f"SELECT * FROM {CATALOG}.{SCHEMA}.silver_fact_shape_points"),
    ("gold_route_groups",             f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_route_groups"),
    ("gold_route_metrics_baseline",   f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_route_metrics_baseline"),
    ("gold_incident_reports",         f"SELECT * FROM {CATALOG}.{SCHEMA}.gold_incident_reports"),
]

skipped = []
for name, query in TABLES:
    try:
        df = spark.sql(query)
        df.write.mode("overwrite").parquet(f"{VOLUME}/{name}")
        print("OK", name, df.count(), "rows")
    except Exception as e:
        skipped.append(name)
        print("SKIP", name, str(e)[:120])

print("Skipped:", skipped if skipped else "none")

In [ ]:
import duckdb

conn = duckdb.connect(f"md:transit?motherduck_token={MD_TOKEN}")

written = [name for name, _ in TABLES if name not in skipped]

for name in written:
    path = f"{VOLUME}/{name}/*.parquet"
    print(f"Loading {name}...", end=" ", flush=True)
    conn.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM read_parquet('{path}')")
    conn.execute(f"SELECT COUNT(*) FROM {name}")
    n = conn.fetchone()[0]
    print(f"{n:,} rows")

# Stamp the export date so the frontend can discover it
conn.execute(f"CREATE OR REPLACE TABLE _export_meta AS SELECT '{yesterday}'::DATE AS data_end_date, now() AS exported_at")

conn.close()
print("MotherDuck sync complete — data through", yesterday)